# Week 4 — Baseline Score (Signals, Rule, Queue)

**Lane, confirmed and locked:** 4 — CTR / Engagement Opportunity Scoring.
I looked hard at Lane 2 again this week since it has the most worked-out example in this
repo, but the tier-vs-gap pattern I found in Weeks 1–2 is still the more interesting
problem to me, so I'm locking Lane 4.

This week's job isn't to build a good model. It's to build one honest baseline — the floor
my Week 5 model has to clear — and to be skeptical of it before I trust it. Two signal
checks first, then one rule, then a hard look at my own top ten.

## Setup

In [1]:
import os, sys, subprocess

IN_COLAB = "google.colab" in sys.modules
REPO_URL = "https://github.com/Hamzakhan0332/flyrank-ml-internship-starter"
REPO_DIR = "flyrank-ml-internship-starter"

if IN_COLAB:
    if not os.path.isdir(REPO_DIR):
        subprocess.run(["git", "clone", "--depth", "1", REPO_URL, REPO_DIR], check=True)
    os.chdir(REPO_DIR)
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", "requirements.txt"], check=True)
elif os.path.basename(os.getcwd()) == "notebooks":
    os.chdir("..")
    os.chdir("..")  # work/notebooks/ -> repo root

sys.path.insert(0, "scripts")
print("Working dir:", os.getcwd())
assert os.path.exists("data/raw/content_refresh_anonymized.csv"), "starter CSV not found — are you at the repo root?"
print("Starter data found.")

Working dir: /content/flyrank-ml-internship-starter
Starter data found.


In [2]:
import pandas as pd
import numpy as np
from ml_utils import percentile_rank  # same helper the starter pipeline uses

df = pd.read_csv("data/raw/content_refresh_anonymized.csv")
df = df[(df["impressions_90d"] > 0) & (df["content_age_days"] >= 90)].drop_duplicates("content_id")

# my lane's slice: real search visibility, competitive position
visible = df[
    (df["impressions_90d"] >= 500) & (df["avg_position"] > 0) & (df["avg_position"] <= 20)
].copy()

tier_expected_ctr = visible.groupby("position_tier")["ctr"].transform("mean")
visible["expected_ctr"] = tier_expected_ctr
visible["ctr_gap"] = visible["ctr"] - visible["expected_ctr"]  # actual minus tier-typical

print(f"Lane 4 candidate pool: {len(visible):,} pages")

Lane 4 candidate pool: 12,023 pages


## 1. Two signal checks + my rule's reasoning

### Signal check 1 — CTR vs. position (the signal behind `needs_ctr_fix`)

FlyRank's product flags a page for a CTR fix when it's underperforming for where it
already ranks. That only makes sense if CTR genuinely differs by position tier in the
first place. Bucket table, with n:

In [3]:
signal1 = visible.groupby("position_tier")["ctr"].agg(
    n="count", mean_ctr="mean", median_ctr="median", std_ctr="std"
).round(3).sort_values("mean_ctr", ascending=False)
signal1

,n,mean_ctr,median_ctr,std_ctr
position_tier,,,,
top_3,458,0.347,0.200,0.422
page_1,7064,0.339,0.240,0.351
striking,4485,0.267,0.170,0.316
page_3_5,16,0.217,0.155,0.246


**Verdict: MIXED.** The tiers do sit in roughly the order I'd expect — `top_3` and
`page_1` cluster around a 0.34–0.35% mean CTR, `striking` distance drops to 0.27%, and
`page_3_5` drops further to 0.22% (though that last bucket only has 16 pages, too thin to
lean on). So the aggregate pattern confirms the assumption underneath `needs_ctr_fix`:
position tier really does set a different bar for "normal" CTR.

But look at the std column next to the mean — for `page_1` the std (0.351) is *bigger*
than the mean itself (0.339). That's not a small caveat, that's the tier telling me
individual pages scatter far more than the tier label predicts. Tier-level, the signal is
real. Page-level, it's noisy enough that I can't use tier alone as a confident per-page
call — it has to be a baseline to compare against, not a verdict on its own. That's exactly
how I use it below: `expected_ctr` per tier as a reference point, not a threshold.

### Signal check 2 — impression volume (the signal behind `is_quick_win`)

My working assumption going in was: more impressions on a low-CTR page means a bigger,
more urgent opportunity, so a "quick win" rule should weight priority up with volume.
I wanted to check that before baking it into a score.

In [4]:
signal2 = visible.groupby("impression_tier")["ctr_gap"].agg(
    n="count", mean_gap="mean", median_gap="median"
).round(4)
signal2

,n,mean_gap,median_gap
impression_tier,,,
excellent,843,0.0267,-0.0668
good,5463,0.0349,-0.0688
moderate,5717,-0.0373,-0.1288


One thing to flag before the verdict: `low` impression tier doesn't show up here at all —
my lane's eligibility rule already requires `impressions_90d >= 500`, which filters every
`low`-tier page out before it ever reaches this table. Good to notice, easy to miss.

**Verdict: OPPOSITE.** I expected impression volume to be roughly independent of
performance — just a dial on how big the opportunity is, not a sign of whether the page is
actually doing fine. Instead, `excellent` and `good` volume pages sit *above* their
position tier's average CTR (positive mean gap), and it's the `moderate` volume band where
the real underperformance concentrates (mean gap -0.037, more negative at the median). In
other words, the highest-traffic pages in my pool already tend to be the ones performing
best for their spot — which is close to backwards from what I assumed going in.

That's a genuine save. If I'd added volume as a straight multiplier on the score like I
originally planned, I'd have been quietly pushing already-fine, high-traffic pages up the
queue and burying the moderate-traffic pages that are the real underperformers. So I'm not
doing that. Volume comes back later, but only as a tiebreaker among pages that are already
tied on the actual opportunity signal — not as a driver of the score itself.

### The rule

One more thing came up while building this that isn't one of the two required checks, but
I don't think it's fair to leave out: **`engagement_rate` is zero for 52.6% of this pool**
(6,324 of 12,023 rows). A field that's exactly zero over half the time doesn't read like
"genuinely no engagement" to me — it reads like a lot of pages never got engagement tracked
at all, and zero is what a missing measurement defaults to. I was going to fold a small
engagement-weakness term into the score. I'm not, because I don't trust that field enough
yet to let it move rankings. Better to leave it out and say why than to quietly build on a
shaky signal.

So the rule ends up simpler than I expected when I started the week:

```text
ctr_engagement_opportunity_score = percentile_rank(-ctr_gap)
```

One driver — how far below its tier's typical CTR a page sits — ranked from worst gap to
smallest. Ties broken by impression volume, descending (biggest audience first, which is
the one place volume earned its way back in, as a tiebreak rather than a weight). One
reason code, `low_ctr_visible_page` — reused verbatim from the starter pipeline's own
naming, since it's literally the eligibility condition every row in this queue shares. One
action label, `review_ctr_and_snippet`.

Every input here — `ctr`, `avg_position`, `position_tier`, `impressions_90d` — is a current
90-day observed signal from the starter CSV. Nothing future-dated, nothing derived from a
FlyRank product decision flag, nothing that's secretly the label in disguise.

## 2. The queue

Score, rank, reason code, action label — then write it out.

In [5]:
visible["ctr_engagement_opportunity_score"] = percentile_rank(-visible["ctr_gap"])
visible["reason_code"] = "low_ctr_visible_page"
visible["action_label"] = "review_ctr_and_snippet"

visible = visible.sort_values(
    ["ctr_engagement_opportunity_score", "impressions_90d"], ascending=[False, False]
).reset_index(drop=True)
visible["baseline_rank"] = visible.index + 1

output_columns = [
    "baseline_rank", "content_id", "client_id", "ctr_engagement_opportunity_score",
    "reason_code", "action_label", "position_tier", "avg_position",
    "impressions_90d", "clicks_90d", "ctr", "expected_ctr", "ctr_gap",
    "engagement_rate", "scroll_rate", "content_type", "main_intent",
]
queue = visible[output_columns]

import pathlib
out_path = pathlib.Path("work/outputs/baseline_action_score.csv")
out_path.parent.mkdir(parents=True, exist_ok=True)
queue.to_csv(out_path, index=False)
print(f"Wrote {len(queue):,} ranked rows to {out_path}")
queue.head(10)

Wrote 12,023 ranked rows to work/outputs/baseline_action_score.csv


,baseline_rank,content_id,client_id,ctr_engagement_opportunity_score,reason_code,action_label,position_tier,avg_position,impressions_90d,clicks_90d,ctr,expected_ctr,ctr_gap,engagement_rate,scroll_rate,content_type,main_intent
0,1,content_50dfd64f9e8e,client_19581e27de,0.997754,low_ctr_visible_page,review_ctr_and_snippet,top_3,2.4,4446,0,0.0,0.346572,-0.346572,0.00,0.00,keyword article,transactional
1,2,content_134631e65b9e,client_6208ef0f77,0.997754,low_ctr_visible_page,review_ctr_and_snippet,top_3,1.5,4417,0,0.0,0.346572,-0.346572,0.00,20.00,keyword article,informational
2,3,content_c90bfc85694f,client_b4944c6ff0,0.997754,low_ctr_visible_page,review_ctr_and_snippet,top_3,1.1,3047,0,0.0,0.346572,-0.346572,0.00,100.00,keyword article,commercial
3,4,content_35d0f98ef4dc,client_19581e27de,0.997754,low_ctr_visible_page,review_ctr_and_snippet,top_3,1.8,2146,0,0.0,0.346572,-0.346572,0.00,0.00,keyword article,informational
4,5,content_4bb993e9270e,client_6208ef0f77,0.997754,low_ctr_visible_page,review_ctr_and_snippet,top_3,1.3,1986,0,0.0,0.346572,-0.346572,0.00,0.00,keyword article,informational
5,6,content_6e6694b67eb6,client_19581e27de,0.997754,low_ctr_visible_page,review_ctr_and_snippet,top_3,1.7,1969,0,0.0,0.346572,-0.346572,0.00,0.00,keyword article,transactional
6,7,content_7d5ad3f9feee,client_19581e27de,0.997754,low_ctr_visible_page,review_ctr_and_snippet,top_3,2.7,1916,0,0.0,0.346572,-0.346572,0.00,0.00,keyword article,commercial
7,8,content_b5964a1a276f,client_19581e27de,0.997754,low_ctr_visible_page,review_ctr_and_snippet,top_3,2.0,1872,0,0.0,0.346572,-0.346572,0.00,25.00,keyword article,informational
8,9,content_56c54a03ece2,client_19581e27de,0.997754,low_ctr_visible_page,review_ctr_and_snippet,top_3,2.7,1638,0,0.0,0.346572,-0.346572,33.33,33.33,keyword article,informational
9,10,content_0ff563db5034,client_6208ef0f77,0.997754,low_ctr_visible_page,review_ctr_and_snippet,top_3,2.2,1563,0,0.0,0.346572,-0.346572,0.00,0.00,keyword article,informational


In [6]:
import json, pathlib

metadata = {
    "lane": "Lane 4 - CTR / Engagement Opportunity Scoring",
    "rows": int(len(queue)),
    "score_formula": "percentile_rank(-ctr_gap), tie-broken by impressions_90d descending",
    "reason_code": "low_ctr_visible_page",
    "action_label": "review_ctr_and_snippet",
    "signal_1_ctr_vs_position": {"verdict": "MIXED", "linked_flag": "needs_ctr_fix"},
    "signal_2_volume": {"verdict": "OPPOSITE", "linked_flag": "is_quick_win"},
    "engagement_rate_excluded_because": "52.6% of the pool is exactly 0, likely missing instrumentation, not a real measurement",
    "top50_position_tier_concentration": queue.head(50)["position_tier"].value_counts().to_dict(),
    "top50_client_concentration_max_share": float(
        queue.head(50)["client_id"].value_counts().iloc[0] / 50
    ),
}
out = pathlib.Path("work/outputs/w04_baseline_metadata.json")
out.write_text(json.dumps(metadata, indent=2))
print(f"Wrote receipts to {out}")
metadata

Wrote receipts to work/outputs/w04_baseline_metadata.json


{'lane': 'Lane 4 - CTR / Engagement Opportunity Scoring',
 'rows': 12023,
 'score_formula': 'percentile_rank(-ctr_gap), tie-broken by impressions_90d descending',
 'reason_code': 'low_ctr_visible_page',
 'action_label': 'review_ctr_and_snippet',
 'signal_1_ctr_vs_position': {'verdict': 'MIXED',
  'linked_flag': 'needs_ctr_fix'},
 'signal_2_volume': {'verdict': 'OPPOSITE', 'linked_flag': 'is_quick_win'},
 'engagement_rate_excluded_because': '52.6% of the pool is exactly 0, likely missing instrumentation, not a real measurement',
 'top50_position_tier_concentration': {'top_3': 50},
 'top50_client_concentration_max_share': 0.4}

## 3. Top-10 review

In [7]:
top10 = queue.head(10).reset_index(drop=True)
for _, row in top10.iterrows():
    print(f"#{row['baseline_rank']:<2} {row['content_id']}  "
          f"tier={row['position_tier']:<8} pos={row['avg_position']:.1f}  "
          f"impr={row['impressions_90d']:>7,}  ctr={row['ctr']:.2f}%  "
          f"gap={row['ctr_gap']:.3f}  intent={row['main_intent']}")

#1  content_50dfd64f9e8e  tier=top_3    pos=2.4  impr=  4,446  ctr=0.00%  gap=-0.347  intent=transactional
#2  content_134631e65b9e  tier=top_3    pos=1.5  impr=  4,417  ctr=0.00%  gap=-0.347  intent=informational
#3  content_c90bfc85694f  tier=top_3    pos=1.1  impr=  3,047  ctr=0.00%  gap=-0.347  intent=commercial
#4  content_35d0f98ef4dc  tier=top_3    pos=1.8  impr=  2,146  ctr=0.00%  gap=-0.347  intent=informational
#5  content_4bb993e9270e  tier=top_3    pos=1.3  impr=  1,986  ctr=0.00%  gap=-0.347  intent=informational
#6  content_6e6694b67eb6  tier=top_3    pos=1.7  impr=  1,969  ctr=0.00%  gap=-0.347  intent=transactional
#7  content_7d5ad3f9feee  tier=top_3    pos=2.7  impr=  1,916  ctr=0.00%  gap=-0.347  intent=commercial
#8  content_b5964a1a276f  tier=top_3    pos=2.0  impr=  1,872  ctr=0.00%  gap=-0.347  intent=informational
#9  content_56c54a03ece2  tier=top_3    pos=2.7  impr=  1,638  ctr=0.00%  gap=-0.347  intent=informational
#10 content_0ff563db5034  tier=top_3    pos

Ten lines, one each — action, why it's here, what would make it wrong:

1. **`content_50dfd64f9e8e`** — action: review title/meta/snippet. Here because it holds
   avg position 2.4 (top_3 tier) with 4,446 impressions and literally 0% CTR. Wrong if: the
   click tracking or a redirect on this URL is broken, since a real page at position 2
   almost never gets zero clicks from 4,400+ impressions — that's more consistent with a
   measurement gap than a content problem.
2. **`content_134631e65b9e`** — action: review title/meta/snippet. Position 1.5, 4,417
   impressions, 0% CTR. Wrong if: this is an informational query where the SERP itself
   (a featured snippet or an answer box) is satisfying the searcher before they ever reach
   a result link — position 1 doesn't guarantee a click if the answer is already on the
   results page.
3. **`content_c90bfc85694f`** — action: review title/meta/snippet. Position 1.1, 3,047
   impressions, 0% CTR, commercial intent. Wrong if: this is a branded/near-duplicate URL
   that shares traffic with a sibling page under the same client, and the real clicks are
   landing there instead of being missing entirely.
4. **`content_35d0f98ef4dc`** — action: review title/meta/snippet. Position 1.8, 2,146
   impressions, 0% CTR, same client as #1 (`client_19581e27de`). Wrong if: the shared
   client explains this better than the shared content problem — a site-wide tagging issue
   would hit several of this client's pages at once, which is worth checking before writing
   five separate content reviews.
5. **`content_4bb993e9270e`** — action: review title/meta/snippet. Position 1.3, 1,986
   impressions, 0% CTR. Wrong if: `content_age_days` shows this page as very new — a page
   that recently earned a top_3 ranking may not have accumulated real click behavior yet,
   and 90 days of near-zero history isn't the same as a settled pattern.
6. **`content_6e6694b67eb6`** — action: review title/meta/snippet. Position 1.7, 1,969
   impressions, 0% CTR, same client as #1 and #4. Third page from this client in the top 6 —
   strengthens the tagging-issue possibility from #4 rather than three independent findings.
7. **`content_7d5ad3f9feee`** — action: review title/meta/snippet. Position 2.7, 1,916
   impressions, 0% CTR, commercial intent, same client again. Wrong if: it's the same
   client-wide explanation as #4 and #6 — at this point that's the more likely story than
   four unrelated content failures.
8. **`content_b5964a1a276f`** — action: review title/meta/snippet. Position 2.0, 1,872
   impressions, 0% CTR, same client once more (fifth appearance in the top 10). Wrong if:
   by now I'd genuinely want to check this client's GA4/GSC integration before touching a
   single page's content.
9. **`content_56c54a03ece2`** — action: review title/meta/snippet. Position 2.7, 1,638
   impressions, 0% CTR — but `engagement_rate` is 33.33 here, not 0 like the rest of the
   top 10. Wrong if: nothing, actually — this is the one row in the top 10 where non-zero
   engagement alongside zero recorded clicks looks like a genuine tracking mismatch (someone
   *engaged* with a page that supposedly got zero clicks), worth pulling first.
10. **`content_0ff563db5034`** — action: review title/meta/snippet. Position 2.2, 1,563
    impressions, 0% CTR, informational intent. Wrong if: same SERP-feature caveat as #2 —
    informational queries in a top_3 slot are the most exposed to an answer box eating the
    click before it happens.

## 4. Weak picks

Two things make me less confident in this top ten than the ranking alone suggests.

**They're not actually ranked against each other.** All ten (and 55 pages total, once I
checked) share the exact same `ctr_gap`, -0.3466 — every one of them has 0% CTR in the
`top_3` tier, so the gap term can't tell them apart at all. My tiebreak (impressions,
descending) is doing all the ordering work inside this group. That's fine for deciding
"which zero gets reviewed first," but it means positions #1 through #10 aren't ten
independently-ranked opportunities — they're one tied group of 55, sorted by audience size.
I'd rather say that plainly than let the rank numbers imply a precision the rule doesn't
have.

In [8]:
min_gap = visible["ctr_gap"].min()
tied = (visible["ctr_gap"] == min_gap).sum()
print(f"Pages tied at the minimum gap ({min_gap:.4f}): {tied}")
print(visible.loc[visible['ctr_gap'] == min_gap, 'main_intent'].value_counts())

Pages tied at the minimum gap (-0.3466): 55
main_intent
informational    30
commercial       15
transactional    10
Name: count, dtype: int64


**One client owns most of the queue.** `client_19581e27de` supplies 5 of my top 10 and
20 of the top 50. That's not necessarily wrong — a client's site could genuinely have
widespread issues — but it's exactly the pattern I'd expect from a broken analytics tag or
a redirect rule affecting many pages at once, not from 20 separate content failures. Before
a reviewer works this queue top-down, I'd want a quick sanity check with that client: is
tracking actually firing correctly on these URLs? If the answer is no, this whole top 50 is
one fix, not fifty.

## 5. Self-check

- [x] Two signal checks, each with a bucket table and n printed: CTR-vs-position (MIXED,
      linked to `needs_ctr_fix`) and impression volume (OPPOSITE, linked to `is_quick_win`).
      Both are flag-linked, though the task only required one.
- [x] The OPPOSITE verdict changed the rule before I wrote it — volume dropped from a score
      weight to a tiebreak only.
- [x] One rule: `percentile_rank(-ctr_gap)`, one reason code (`low_ctr_visible_page`), one
      action label (`review_ctr_and_snippet`).
- [x] Ranked queue written to `work/outputs/baseline_action_score.csv` from this notebook,
      and a metadata JSON receipt written to `work/outputs/w04_baseline_metadata.json`.
- [x] Ten rows reviewed individually — action, why it's there, what would make it wrong.
- [x] Weak picks named honestly: the top 10 are a tied group, not ten independent ranks,
      and one client dominates the queue in a way that smells more like a tracking issue
      than fifty separate content problems.
- [x] No future-window fields, no product decision flags, no label leakage — every input is
      an observable current-window signal from the starter CSV.
- [ ] Still open before Week 5: I excluded `engagement_rate` for data-quality reasons
      rather than testing it properly. If I can find a way to distinguish "genuinely zero
      engagement" from "never tracked," it might be worth reintroducing.